# Aplicações da Inteligência Artificial

Prof. [Denilson Alves Pereira](https://sites.google.com/ufla.br/denilsonpereira) <br>
Departamento de Ciência da Computação (DCC) <br>
Instituto de Ciências Exatas e Tecnológicas (ICET) <br> 
Universidade Federal de Lavras (UFLA)

## RAG - Retrieval-Augmented Generation (Geração Aumentada de Recuperação)

Objetivo: apresentar o conceito de RAG e um exemplo simples de implementação.

Você vai aprender a:
- Aplicar os conceitos de RAG em uma implementação básica (do zero)
- Usar um modelo de linguagem para codificar, na forma de embeddings, uma base de conhecimento textual
- Criar um banco de dados vetorial para armazenar os embeddings da base de conhecimento
- Gerar o contexto para responder consultas do usuário a partir dos embeddings da base de conhecimento
- Usar um modelo de linguagem pré-treinados para responder as consultas do usuário

Referências: <br>
https://huggingface.co/blog/ngxson/make-your-own-rag <br>
https://aws.amazon.com/pt/what-is/retrieval-augmented-generation/

Versão: Janeiro, 2025

## Conceito de RAG

Os Grandes Modelos de Linguagem (LLM - *Large Language Model*) são treinados em grandes volumes de dados e podem ser usados para tarefas como responder perguntas e traduzir idiomas. Entretanto, eles são genéricos, e podem não ter conhecimentos detalhados em domínios específicos ou da base de dados da sua empresa.

Retrieval-Augmented Generation (RAG) é o processo de usar um LLM para responder consultas de usuários fazendo referência a uma base de conhecimento fora das fontes de dados em que ele foi treinado. RAG combina recuperação de informação com geração de texto. Dessa forma, é possível responder perguntas acessando informações atualizadas ou de domínio específico.

<img src="./figs/rag-fluxograma.jpg" width="720" height="400" alt="RAG Fluxograma"> <br>
Referência: https://aws.amazon.com/pt/what-is/retrieval-augmented-generation/

## Componentes de um Sistema de RAG

Um sistema de RAG consiste de dois componentes principais:
- Um modelo de recuperação que busca informação relevante de uma fonte de conhecimento externo, a qual pode ser um banco de dados, uma máquina de busca ou qualquer outro repositório de informação.
- Um modelo de linguagem (LLM) que gera respostas baseadas no conhecimento recuperado.

## RAG Simples

Vamos criar um sistema de RAG simples que recupera informação de um dataset predefinido e gera respostas baseadas no conhecimento recuperado. O sistema compreenderá os seguintes componentes:
1. Modelo de Embedding: um modelo de linguagem predefinido que converte textos de entrada em embeddings (representação vetorial que captura o significado semântico). Esses vetores serão usados para pesquisar informação relevante no dataset.
2. Banco de Dados Vetorial: Um sistema de armazenamento para a base de conhecimento e seus correspondentes vetores de embedding. Existem várias tecnologias para bancos de dados vetoriais, tais como FAISS, pgvector, Qdrant e Pinecone. Nesse exemplo, nós implementaremos um banco de dados simples em memória.
3. Chatbot: Um modelo de linguagem que gera respostas baseadas no conhecimento recuperado. Esse pode ser qualquer modelo de linguagem, tais como Llama, Gemma ou GPT.

<img src="./figs/rag-simples.svg" width="240" height="140" alt="RAG simples"> <br>
Referência: https://huggingface.co/blog/ngxson/make-your-own-rag

## Fase de Indexação

A indexação é o primeiro passo na criação de um sistema de RAG. Ela envolve quebrar os documentos em blocos (*chunks*) menores e calcular a representação vetorial de cada bloco, de forma que possam ser eficientemente buscados durante a geração.

<img src="./figs/rag-indexacao.svg" width="400" height="240" alt="RAG Indexação"> <br>
Referência: https://huggingface.co/blog/ngxson/make-your-own-rag

O tamanho de cada bloco pode variar dependendo da base de dados e da aplicação. Por exemplo, em um sistema de recuperação de documentos, cada bloco pode ser um parágrafo ou uma sentença. Em um sistema de diálogo, cada bloco pode ser o trecho da mudança da conversa.

Depois da fase de indexação, cada bloco com seu correspondente vetor de embedding será armazenado no banco de dados vetorial.

## Fase de Recuperação

Dada como entrada a consulta do usuário, o sistema calcula a representação vetorial dessa consulta e a compara com os vetores no banco de dados para encontrar os blocos mais relevantes. Para comparar a similaridade entre os vetores de embeddings, consulta e cada bloco armazenado, nós podemos usar a Similaridade do Cosseno, a Distância Euclidiana, ou outras métricas de distância.

O resultado retornado pelo banco de dados vetorial conterá os top N blocos mais relevantes para a consulta. Esses blocos serão usados pelo Chatbot para gerar a resposta.

<img src="./figs/rag-recuperacao.svg" width="300" height="180" alt="RAG Recuperação"> <br>
Referência: https://huggingface.co/blog/ngxson/make-your-own-rag

## Fase de Geração

Os blocos relevantes recuperados na fase de recuperação são usados para alimentar o LLM com o contexto para gerar respostas coerentes, baseadas nos fatos da base de conhecimento. Para isso, o LLM usa tanto o contexto quanto seu conhecimento interno.

#### Fluxograma das três fases:

<img src="./figs/rag-fluxograma.webp" width="600" height="480" alt="RAG Fluxograma"> <br>
Referência: https://medium.com/@danushidk507/rag-with-llama-using-ollama-a-deep-dive-into-retrieval-augmented-generation-c58b9a1cfcd3

## Código em Python para um Sistema de RAG Simples

#### Ollama

Faça o download da ferramenta Ollama. Ela é um framework que permite que você execute modelos diretamente no seu computador, sem a necessidade de acessar um servidor ou serviço em nuvem.

Instale o Ollama a partir do website do projeto: https://ollama.com/

Depois, abra um terminal e execute os seguintes comandos para fazer o download dos modelos requeridos: <br>
ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf <br>
ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF

Finalmente, para usar o Ollama em Python, instale o pacote: <br>
pip install ollama

#### Llama

Para o LLM do chatbot, vamos usar o Llama, um modelo desenvolvido pela Meta (Facebook). Ele foi liberado sob uma licença 'open access', e pode ser usado localmente em seu servidor, sem a necessidade de pagar o acesso via uma API. Llama está disponível em vários tamanhos (quantidade de parâmetros), permitindo o desenvolvedor escolher o modelo baseado em seu caso de uso e restrições computacionais.

#### Ambiente

Crie um ambiente para o projeto. <br>
Execute o código abaixo em um terminal do sistema operacional.

In [1]:
# Baixe o Ollama de https://ollama.com/

# Baixe os modelos a serem usados no exemplo:
# ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
# ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF

# Crie um ambiente para o projeto
# python3 -m venv $HOME/rag-env
# source $HOME/rag-env/bin/activate
# pip install jupyter

# Instale o seguinte pacote
# pip install ollama

### Base de Conhecimento

O dataset 'cat facts' contém uma lista de fatos sobre gatos. Cada uma de suas entradas será usada como bloco (chunk) na fase de indexação.<br>
O dataset pode ser baixado do link: https://huggingface.co/ngxson/demo_simple_rag_py/blob/main/cat-facts.txt

In [2]:
# Carrega o dataset
dataset = []
with open('./datasets/cat-facts.txt', 'r') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')

Loaded 150 entries


### Banco de Dados Vetorial

Implementação do banco de dados vetorial em memória, por simplicidade. <br>
Cada elemento no banco de dados será uma tupla (chunk, embedding)

In [3]:
# Implementação do banco de dados vetorial

import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Cada elemento no banco de dados será uma tupla (chunk, embedding)
# O embedding é um lista de números, por exemplo: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

In [4]:
# Cada linha no dataset será considerada como um bloco (chunk), por simplicidade
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')

Added chunk 1/150 to the database
Added chunk 2/150 to the database
Added chunk 3/150 to the database
Added chunk 4/150 to the database
Added chunk 5/150 to the database
Added chunk 6/150 to the database
Added chunk 7/150 to the database
Added chunk 8/150 to the database
Added chunk 9/150 to the database
Added chunk 10/150 to the database
Added chunk 11/150 to the database
Added chunk 12/150 to the database
Added chunk 13/150 to the database
Added chunk 14/150 to the database
Added chunk 15/150 to the database
Added chunk 16/150 to the database
Added chunk 17/150 to the database
Added chunk 18/150 to the database
Added chunk 19/150 to the database
Added chunk 20/150 to the database
Added chunk 21/150 to the database
Added chunk 22/150 to the database
Added chunk 23/150 to the database
Added chunk 24/150 to the database
Added chunk 25/150 to the database
Added chunk 26/150 to the database
Added chunk 27/150 to the database
Added chunk 28/150 to the database
Added chunk 29/150 to the dat

### Função de Recuperação

A função de recuperação recebe como entrada a consulta do usuário e retorna os top N chunks mais relevantes, baseado na Similaridade do Cosseno. Quanto mais alta a similaridade do cosseno entre dois vetores, mais próximos eles estão no espaço vetorial. Isso significa que eles são mais similares em termos de significado.

In [5]:
# Similaridade do Cosseno entre dois vetores
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [6]:
# Função de Recuperação
def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # lista temporária para armazenar pares (chunk, similaridade)
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # ordena por similaridade em ordem decrescente, pois similaridades mais altas significam chunks mais relevantes
  similarities.sort(key=lambda x: x[1], reverse=True)
  # retorna os top N chunks mais relevantes
  return similarities[:top_n]

### Fase de Geração

Nesta fase, o chatbot vai gerar a resposta baseada no conhecimento recuperado na fase anterior. Isso é feito simplesmente adicionando-se os chunks dentro do prompt que será levado como entrada para o chatbot.

Por exemplo, um prompt pode ser contruído como a seguir:

In [8]:
# Fase de Geração

# prompt
input_query = input('Faça-me uma pergunta (em inglês): ')
retrieved_knowledge = retrieve(input_query)

print('Conhecimento recuperado:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similaridade: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:\n
{''.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}'''

Faça-me uma pergunta (em inglês):  tell me about cat speed


Conhecimento recuperado:
 - (similaridade: 0.78) A cat can travel at a top speed of approximately 31 mph (49 km) over a short distance.

 - (similaridade: 0.66) Cats are extremely sensitive to vibrations. Cats are said to detect earthquake tremors 10 or 15 minutes before humans can.

 - (similaridade: 0.66) Cats sleep 16 to 18 hours per day. When cats are asleep, they are still alert to incoming stimuli. If you poke the tail of a sleeping cat, it will respond accordingly.



Depois disso, usamos o Ollama para gerar a resposta.

In [9]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# Imprime a resposta do chatbot em tempo real
print('Resposta do chatbot:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Resposta do chatbot:
Cats are known for their impressive agility and quick movements. According to the context, a cat can travel at approximately 31 mph (49 km) over a short distance. This makes them one of the fastest animals on four legs. However, it's worth noting that this is just an estimate, as exact speed records for cats are not always publicly available.

## Comentários Finais

Esse notebook apresentou os conceitos básicos de RAG e uma implementação simples. Há ainda muito espaço para melhorias, tais como quebrar consultas complexas em tópicos, re-ranquear os chunks recuperados, usar um banco de dados vetorial eficiente, usar técnicas mais sofisticadas para quebrar os documentos em chunks, usar um modelo de linguagem com maior capacidade (maior número de parâmetros). Além disso, pode-se representar o conhecimento por meio de grafos (Graph RAG). RAG é um tópico de pesquisa em constante evolução. Para se obter bons resultados, é preciso estar atento ao estado-da-arte na pesquisa.